# Dataset Setup

One-Time only run-cells are changed to markdown-cells, to avoid overwritting existing processed files.

#### Requirements
- smplx
- numpy
- scipy
- chumpy

In [1]:
from pathlib import Path

DATA_DIR = Path.cwd() / "datasets"
AIST_DIR = DATA_DIR / "AIST"
SMPL_DIR = DATA_DIR  / "SMPL"
AIST_DIR.mkdir(parents=True, exist_ok=True)

## Refernces:
1. [Data Formats](https://google.github.io/aistplusplus_dataset/download.html)
2. [SMPL Format](https://medium.com/@bhipanshudhupar/what-is-smpl-the-3d-human-body-model-powering-modern-ai-and-animation-c654a0284800)
3. [SMPL made simple FAQs](https://files.is.tue.mpg.de/black/talks/SMPL-made-simple-FAQs.pdf)

FYI:
- **motions** store SMPL parameters only (refer 6D rotational representation in paper),i.e., R^{S×(24×6+4+3)} where J3D=24 (each 6 param), foot contact labels = R^4, root pos = R^3.

- **keypoints2d:** Multi-view frame-by-frame 2D keypoints detection results. Array shape is (9, N, 17, 3) where
    - The first dim represents Individual environment settings, each with 9 cameras.
    - J2D = 17 (no.of joints in 2d repr. w.r.t coco semantics).
    - The last dim, i.e. each joint contains (x, y, confidence).
    - **NOTE:** keypoints are given directly in image pixel coordinates, not normalized or made relative to a smaller scale. The AIST++ videos are 1080p resolution, with frames sized at 1920×1080 pixels — the keypoints are not scaled
    
- **keypoints3d:** Raw reconstructed 3D joint coordinates (i.e. last dim is 3D: x,y,z) and smoothed/optimized versions (keypoints3d_optim) plus metadata. Array shape is (N, 17, 3).

- **Timestamps:** Annotations are frame-by-frame under exact 60 FPS. Some videos in the AIST Dance Video DB have slightly different FPS, but hard-coded 60 FPS when converting videos into images.

- Canonical **SMPL joint id** → name mapping used by SMPL/SMPL-X:

|  |  |  |
| --- | --- | --- |
| 0:  Pelvis    | 1:  L_Hip     | 2:  R_Hip     |
| 3:  Spine1    | 4:  L_Knee    | 5:  R_Knee    | 
| 6:  Spine2    | 7:  L_Ankle   | 8:  R_Ankle   |
| 9:  Spine3    | 10: L_Foot    | 11: R_Foot    |
| 12: Neck      | 13: L_Collar  | 14: R_Collar  |
| 15: Head      | 16: L_Shoulder| 17: R_Shoulder|
| 18: L_Elbow 	| 20: L_Wrist 	| 22: L_Hand 	|
| 19: R_Elbow 	| 21: R_Wrist 	| 23: R_Hand 	|


In [2]:
mocap_zip = AIST_DIR / "motions.zip"
mocap_dir = mocap_zip.with_suffix('')
keypoints_zip = AIST_DIR / "keypoints2d.zip"
keypoints_dir = keypoints_zip.with_suffix('')

```python
# Check and download motions.zip
if not mocap_dir.exists():
    if not mocap_zip.exists():
        print(f"File {mocap_zip.name} does not exist. Downloading...")
        !wget -c -q https://storage.googleapis.com/aist_plusplus_public/20210308/motions.zip -O {mocap_zip} \
        || echo "Failed to download {mocap_zip}!"
    else:
        print(f"File {mocap_zip.name} already exists. Skipping download.")
    !unzip -q {mocap_zip} -d {AIST_DIR} || echo "{mocap_zip} unzip failed"
print(f"Download & Unzip checks completed in '{mocap_dir}' :)")

# Check and download keypoints2d.zip
if not keypoints_dir.exists():
    if not keypoints_zip.exists():
        print(f"File {keypoints_zip.name} does not exist. Downloading...")
        !wget -c -q https://storage.googleapis.com/aist_plusplus_public/20210308/keypoints2d.zip -O {keypoints_zip} \
        || echo "Failed to download {keypoints_zip}!"
    else:
        print(f"File {keypoints_zip.name} already exists. Skipping download.")
    !unzip -q {keypoints_zip} -d {AIST_DIR} || echo "{keypoints_zip} unzip failed"
print(f"Download & Unzip checks completed in '{keypoints_dir}' :)")

```

In [3]:
# quick test: open one motion pkl and one keypoints2d pkl to verify fields
import pickle, glob
M3D_pkl = glob.glob(f"{mocap_dir}/*.pkl")
P3D_pkl = glob.glob(f"{keypoints_dir}/*.pkl")

print("Found motion files:", len(M3D_pkl), "Found keypoints2d files:", len(P3D_pkl))

if M3D_pkl:
    with open(M3D_pkl[0], "rb") as f:
        m = pickle.load(f)
    print("\nMotion keys:", list(m.keys()))
    for k,v in m.items():
        print(f"{k} shape:", v.shape if hasattr(v, 'shape') else type(v), end=", ")
    print()

if P3D_pkl:
    with open(P3D_pkl[0], "rb") as f:
        p = pickle.load(f)
    print("\nKeypoints keys:", list(p.keys()))
    for k,v in p.items():
        print(f"{k} shape:", v.shape if hasattr(v, 'shape') else type(v), end=", ")
    print()

Found motion files: 1408 Found keypoints2d files: 1510

Motion keys: ['smpl_loss', 'smpl_poses', 'smpl_scaling', 'smpl_trans']
smpl_loss shape: <class 'float'>, smpl_poses shape: (720, 72), smpl_scaling shape: (1,), smpl_trans shape: (720, 3), 

Keypoints keys: ['keypoints2d', 'det_scores', 'timestamps']
keypoints2d shape: (9, 720, 17, 3), det_scores shape: (9, 720), timestamps shape: (720,), 


Note: each data-entry: no.of time-stamps: 480 => 8 secs of 60 fps motions

### Preprocess & pack dataset for ViMo training
- Resamples 60 → 30 FPS by taking every 2nd frame (paper re-aligns to 30 FPS for training).
- Cuts sequences into 5s clips (S = 150 frames).
- Keeps cond_2d as views x 150 x 17 x 3 (multi-view 2D detections).
- Builds m3d_gt per clip as 150 x 151 (24×6 rotations + 4 foot contact + 3 root).
- Skips sequences listed in ignore_list.txt if present.
- How do I convert SMPL part rotations to other formats:
    - [pytorch3d](https://github.com/facebookresearch/pytorch3d): <br>
	[dependency issues](https://github.com/facebookresearch/pytorch3d/issues/1970#issuecomment-2858723992): thus use versions below:
	```py
	!pip install torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu128 --force-reinstall --no-cache-dir
	```
    - refer page 32 at "SMPL made simple FAQ"

In [4]:
# convert AIST++ annotations -> ViMo training samples (.npz)
import numpy as np

AIST_OUT = AIST_DIR / "processed"   # output ViMo-ready dataset
S = 150   # frames per clip (5s * 30 fps)
SRC_FPS = 60
TARGET_FPS = 30
DOWNSAMPLE = SRC_FPS // TARGET_FPS  # should be 2

# Foot joint ids for SMPL (refer above)
ANKLE_IDX = (7, 8)   # L_Ankle, R_Ankle
FOOT_IDX  = (10,11)  # L_Foot,  R_Foot

In [5]:
'''
# utility: axis-angle -> rotation matrices (vectorized Rodrigues)
def axisangle_to_rotmat_batch(axisangle):
    # input: axisangle=(S,24,3); output: Rmat=(S,24,3,3)
    S = axisangle.shape[0]
    assert axisangle.ndim == 3 # case: (S, 72) need to be reshaped already
    R = np.zeros((S, 24, 3, 3), dtype=np.float32)
    for t in range(S):
        for j in range(24):
            v = axisangle[t,j]
            theta = np.linalg.norm(v)
            if theta < 1e-8:
                R[t,j] = np.eye(3, dtype=np.float32)
            else:
                k = (v / theta).astype(np.float32)
                K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]], dtype=np.float32)
                R[t,j] = np.eye(3, dtype=np.float32) + np.sin(theta)*K + (1-np.cos(theta))*(K @ K)
    return R

def rotmat_to_6d(R):
    # R: (...,3,3) -> take first two cols -> (...,6)
    cols = R[..., :2]  # (...,3,2)
    return cols.reshape(*R.shape[:-2], 6)
'''
from pytorch3d import transforms # contains useful 3D transform utils as above

# utility: foot contact from joint positions
def compute_foot_contact_from_joints(joint_pos, ankle_idx=ANKLE_IDX, foot_idx=FOOT_IDX, vel_thresh=1e-3):
    """
    joint_pos: (S, J, 3) -- SMPL joints in world coordinates (J=24)
    returns: contacts: (S,4) int8 corresponding to:
             [L_ankle_contact, R_ankle_contact, L_foot_contact, R_foot_contact]
    """
    S = joint_pos.shape[0]
    contacts = np.zeros((S,4), dtype=np.int8)
    vel = np.zeros_like(joint_pos) # vel[0] stays zero
    vel[1:] = joint_pos[1:] - joint_pos[:-1] # velocity at frame t is pos[t] - pos[t-1].
    # vertical, i.e. up-axis (0:x,1:y,2:z) assumed y; threshold on y-comp-vel small => contact
    for t in range(1, S):
        contacts[t,0] = int(abs(vel[t, ankle_idx[0], 1]) < vel_thresh) # left ankle
        contacts[t,1] = int(abs(vel[t, ankle_idx[1], 1]) < vel_thresh) # right ankle
        contacts[t,2] = int(abs(vel[t, foot_idx[0], 1]) < vel_thresh) # left foot
        contacts[t,3] = int(abs(vel[t, foot_idx[1], 1]) < vel_thresh) # right foot
    return contacts

#### fix getargspec deprecation in python 3.11
This [error](https://github.com/mattloper/chumpy/pull/59) occurs because inspect.getargspec() was removed in Python 3.11, after being deprecated for many years. The chumpy library (used by SMPL and SMPL-X models) still calls this function, which triggers the AttributeError. //@bugfix-commit-id

Fix: `pip install git+https://github.com/mattloper/chumpy@9b045ff5d6588a24a0bab52c83f032e2ba433e17`
Add attrib: '--no-build-isolation' when building directly in core env

In [6]:
# ---------- Try to load SMPL model ----------
from pathlib import Path
import os, re, glob, smplx, torch

smpl_files = glob.glob(f"{SMPL_DIR}/v*/smpl/*.pkl")
print("Found candidate .pkl files:")
for file in smpl_files:
    print(" -", file, f"({os.path.getsize(file)/1e6:.2f} MB)")
if len(smpl_files) == 0:
    print("Please download and rename SMPL models (licensed) from https://smpl.is.tue.mpg.de/")
    raise FileNotFoundError(f"No SMPL .pkl files found under {SMPL_DIR}")
else:
    print("OK — some .pkl files found. Next step: Attempt to load.")

def version_tuple_from_path(f, iter=2):
    # infer version from parent folder name like "v1.0.0" or "v1.1.0"
    p = Path(f)
    for i in range(iter):
        p = p.parent
    parent = p.name
    nums = re.findall(r"\d+", parent)
    if not nums:
        return (0,)
    return tuple(int(x) for x in nums)

# Prefer a neutral model if present (case-insensitive); otherwise fall back to first candidate
neutral_types = [file for file in smpl_files if "neutral" in os.path.basename(file).lower()]
if len(neutral_types) > 0:
    chosen = max(neutral_types, key=version_tuple_from_path)
else:
    # fallback: pick highest-version candidate regardless of gender
    chosen = max(smpl_files, key=version_tuple_from_path)
chosen_pkl = Path(chosen)
model_folder = chosen_pkl.parent.parent
print(f"Chosen SMPL model.pkl: {chosen_pkl.name} | folder: {model_folder.name}")

SMPL_MODEL = None
# Create SMPL model using the neutral gender option
try:
    SMPL_MODEL = smplx.create(model_path=model_folder, model_type='smpl', gender='neutral', use_pca=False)
    print(f"SMPL model created with {SMPL_MODEL.J_regressor.shape[0]} joints.")
except Exception as e:
    # If smplx.create fails, re-raise with helpful debug info
    raise RuntimeError(f"smplx.create failed for model_folder={model_folder} with error: {e}")
            

Found candidate .pkl files:
 - d:\STUDIES\MTech\#MTP\codes\vimo\datasets\SMPL\v1.0.0\smpl\SMPL_FEMALE.pkl (39.06 MB)
 - d:\STUDIES\MTech\#MTP\codes\vimo\datasets\SMPL\v1.0.0\smpl\SMPL_MALE.pkl (39.06 MB)
 - d:\STUDIES\MTech\#MTP\codes\vimo\datasets\SMPL\v1.1.0\smpl\SMPL_FEMALE.pkl (247.53 MB)
 - d:\STUDIES\MTech\#MTP\codes\vimo\datasets\SMPL\v1.1.0\smpl\SMPL_MALE.pkl (247.10 MB)
 - d:\STUDIES\MTech\#MTP\codes\vimo\datasets\SMPL\v1.1.0\smpl\SMPL_NEUTRAL.pkl (247.19 MB)
OK — some .pkl files found. Next step: Attempt to load.
Chosen SMPL model.pkl: SMPL_NEUTRAL.pkl | folder: v1.1.0
SMPL model created with 24 joints.


In [7]:
def preprocess_clip(name, pose2d, j_axis3d, root3d, verbose=False):
	''' j_axis3d: standard=(S,24,3) or from dataset=(S,72) '''
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	S = j_axis3d.shape[0]
	if j_axis3d.ndim == 2: # case: (S, 72)
		j_axis3d = j_axis3d.reshape(S, 24, 3)

	if not torch.is_tensor(j_axis3d): j_axis3d = torch.tensor(j_axis3d, device=device).float()
	if not torch.is_tensor(root3d): root3d = torch.tensor(root3d, device=device).float()
	SMPL_MODEL.to(device)
	if verbose: print(f"\t>> Clip-{name}: p2d.shape={pose2d.shape}, j_axis3d.shape={j_axis3d.shape}, root3d.shape={root3d.shape})", end=" ")

	# convert joints axis-angle -> rotmatrix -> 6d representation
	Rmats = transforms.axis_angle_to_matrix(j_axis3d)  # (S,24,3,3) prev, axisangle_to_rotmat_batch(j_axis3d)
	j6d = transforms.matrix_to_rotation_6d(Rmats)      # (S,24,6) prev, rotmat_to_6d(Rmats) 
	j6d_flat = j6d.reshape(S, -1)              # (S, 144)
	if verbose: print(f", j_Rmats.shape={Rmats.shape}, j6d.shape={j6d.shape}", end=" ")

	max_abs = root3d.abs().max().item() if torch.is_tensor(root3d) else np.abs(root3d).max()
	is_normalized = max_abs < 3.0 # Heuristic: scale-normalized if max_abs < 3
	if not is_normalized: 
		if verbose: print(f"WARNING: Given input is not scale normalized (max-abs of root-pos:{max_abs} must be less than {3.0}),",
				"applying smpl.fwd(without 'transl' param), i.e. root pos is fixed at orgin for all frames")

	# compute joint world positions with SMPL FK if available
	j_worldPos = None
	try:
		# smplx expects global_orient (B,3), body_pose (B,69), transl (B,3)
		# call in batches (smpl can process batch) To feed these into the SMPL model, we have to split the data into:
		# global_orient — rotation of the root joint only (pelvis:joint-id=0)
		# body_pose — rotations of all other 23 joints
		# transl — translation vector for the whole body
		with torch.no_grad():
			go = j_axis3d[:, 0, :]                        # (S,1,3)=>(S,3)
			body = j_axis3d[:, 1:, :].reshape(S, 23*3)    # (S,23,3)=>(S,69)
			tr = root3d                                   # (S,3)
			#if verbose: print(f", global_orient.shape={go.size()}, body_pose.shape={body.size()}, transl.shape={tr.size()}", end=" ")
			if is_normalized:
				output = SMPL_MODEL(global_orient=go, body_pose=body, transl=tr)
			else:
				output = SMPL_MODEL(global_orient=go, body_pose=body)
			# output.joints shape (S, J, 3) where J >= 24. We'll take the first 24 SMPL joints.
			#if verbose: print(f"SMPL FK output.2
			#if verbose: print(f"BodyPose.shape={output.body_pose.shape}")
			#if verbose: raise RuntimeError("CheckPoint")
			joints = output.joints.cpu().numpy()   # (S, J, 3)
			j_worldPos = joints[:, :24, :]       # (S,24,3)
		if verbose: print(f", j_worldPos.shape={j_worldPos.shape}", end=" ")
	except Exception as e:
		raise RuntimeError("\nSMPL FK failed for", name, "->. Error:", e)

	# compute contacts
	contacts = np.zeros((S,4), dtype=np.int8)
	if j_worldPos is not None:
		contacts = compute_foot_contact_from_joints(j_worldPos)
		if verbose: print(f", contacts.shape={contacts.shape}", end=" ")
	else:
		# fallback: use root vertical velocity proxy (not exact); produce 4 dims by duplicating left/right
		vel = np.zeros_like(root3d)
		vel[1:] = root3d[1:] - root3d[:-1] # velocity at frame t is root_pos[t] - root_pos[t-1].
		speed = np.linalg.norm(vel, axis=1)              # (S,) use full 3D speed (norm) instead of only y-component
		c = (speed < 1e-2).astype(np.int8)               # contact proxy by speed threshold
		contacts = np.repeat(c[:, None], 4, axis=1)      # make 4-channel contact
		print(f"WARNING: using root-velocity proxy contacts for {name} (SMPL models not available or FK failed).")

	# ground truth 3D motion (S,151), i.e. target motion: concat j6d_flat (144) + contacts (4) + root (3) -> 151
	m3d_gt = np.concatenate([j6d_flat.cpu().numpy(), contacts, root3d.cpu().numpy()], axis=1)  

	# save as .npz: contains 'p2d_cond' (views, S, 17, 3) and 'm3d_gt' (S,151)
	out_name = f"{name}.npz"
	np.savez_compressed(os.path.join(AIST_OUT, out_name), p2d_cond=clip_p2d.astype(np.float32), m3d_gt=m3d_gt.astype(np.float32))
	if verbose: print(f", m3d_gt.shape={m3d_gt.shape}, Saved successfully!")


In [9]:
#```python
#from tqdm import tqdm
AIST_OUT.mkdir(parents=True, exist_ok=True)
verbose = False
np.set_printoptions(suppress=True, precision=6, floatmode='fixed')  # fixed-point, no exponent

# iterate sequences, pair motion and keypoints by filename prefix
count_saved = 0
for m in M3D_pkl:
	name = Path(m).stem
	p = os.path.join(keypoints_dir, name + ".pkl")
	if not os.path.exists(p): # skip if not found
		continue

	pdata = pickle.load(open(p, "rb"))
	p2d_all = pdata.get("keypoints2d", None)
	if p2d_all is None:
		continue
	mdata = pickle.load(open(m, "rb"))
	poses = mdata.get("smpl_poses", None)    # axis-angle per joint
	trans = mdata.get("smpl_trans", None)    # root translation or global position
	scale = mdata.get("smpl_scaling", 1.0) # scalar, extract scaling (def=1,i.e. norm form)
	if poses is None or trans is None:
		continue

	# downsample from 60->30 fps by selecting every DOWNSAMPLE-th frame
	p2d_ds = p2d_all[:, ::DOWNSAMPLE]   	# shape (views, N_ds, 17, 3)
	poses_ds = poses[::DOWNSAMPLE]  		# (N_ds,24,3)
	trans_ds = trans[::DOWNSAMPLE] / scale  # (N_ds,3) Normalize to canonical scale
	assert p2d_ds.shape[1] == poses_ds.shape[0] == trans_ds.shape[0]
	N_ds = poses_ds.shape[0]

	# cut into S-length clips
	num_clips = int(np.ceil(N_ds / S))
	multiples = N_ds / max(1, num_clips - 1) # avoid div by zero
	if verbose: print(f"Processing {name}: total frames {N_ds}, num_clips {num_clips} ...")

	if N_ds < S: # pad if needed by repeating last frame to reach S
		st, ed = 0, N_ds
		need = S - N_ds
		last_p2d = p2d_ds[:, -1:, :, :].repeat(need, axis=1)
		clip_p2d = np.concatenate([p2d_ds, last_p2d], axis=1)
		last_j3d = poses_ds[-1:, :, :].repeat(need, axis=0)
		clip_j3d = np.concatenate([poses_ds, last_j3d], axis=0)
		last_root = trans_ds[-1:, :].repeat(need, axis=0)
		clip_root3d = np.concatenate([trans_ds, last_root], axis=0)
		
		clip_name = f"{name}_clip[{st:04d}-{ed:04d}]"
		preprocess_clip(clip_name, clip_p2d, clip_j3d, clip_root3d, verbose)
		count_saved += 1

	else:
		low = 0; high = N_ds
		for i in range(num_clips):
			j = i//2; r=i%2
			if r==0:
				if i==num_clips-1: # last midmost clip
					mid = low + int(np.ceil((high-low)/2))
					st = mid - S//2
				else:
					st = low
				ed = st+S
				low = ed
			else:
				ed = high
				st = ed-S
				high = st
			
			# p2d_cond: keep all views (9) as paper used multi-view projected 2D
			clip_p2d = p2d_ds[:, st:ed]   # (views, tlen, 17, 3)
			clip_j3d = poses_ds[st:ed]   # (tlen,24,3)
			clip_root3d = trans_ds[st:ed]

			clip_name = f"{name}_clip[{st:04d}-{ed:04d}]"
			preprocess_clip(clip_name, clip_p2d, clip_j3d, clip_root3d, verbose)
			#print("actual trans:\n", trans[st:ed:DOWNSAMPLE], f"\nscaled down by {scale}:\n", clip_root3d, "\n")
			count_saved += 1
	if verbose: print()

print("Totally Saved", count_saved, "ViMo-ready clips to", AIST_OUT)
#```
#Expected Output: # Totally Saved 4355 ViMo-ready clips to d:\STUDIES\MTech\#MTP\codes\vimo\datasets\AIST\processed

Totally Saved 4355 ViMo-ready clips to d:\STUDIES\MTech\#MTP\codes\vimo\datasets\AIST\processed


In [10]:
import glob, random, numpy as np
ex = glob.glob(f"{AIST_OUT}/*.npz")
if not ex:
    print("No saved ViMo dataset samples found. Run the previous cell.")
else:
    #sample_id = random.randint(0, len(ex)-1)
	sample_id = 3
	data = np.load(ex[sample_id])
	print(f"Loaded [{sample_id}]: {ex[sample_id]}")
	ex_m3d = data['m3d_gt']
	frame_idx = S//2
	print(f"p2d_cond.shape (views,S,17,3): {data['p2d_cond'].shape} || m3d_gt.shape (S,151): {ex_m3d.shape}")
	print(f"One frame of m3d_gt: \nJoints:{ex_m3d[frame_idx,:144]} \nContacts:{ex_m3d[frame_idx,144:148]} \nRoot:{ex_m3d[frame_idx,148:151]}")

Loaded [3]: d:\STUDIES\MTech\#MTP\codes\vimo\datasets\AIST\processed\gBR_sBM_cAll_d04_mBR0_ch02_clip[0000-0150].npz
p2d_cond.shape (views,S,17,3): (9, 150, 17, 3) || m3d_gt.shape (S,151): (150, 151)
One frame of m3d_gt: 
Joints:[ 0.762645  0.215461  0.609877 -0.261074  0.965209 -0.014524  0.970253
 -0.241995  0.006890  0.227876  0.922513  0.311515  0.916420  0.058875
  0.395863 -0.191846  0.932696  0.305405  0.995828  0.083527  0.036728
 -0.075131  0.979017 -0.189425  0.957862  0.285446 -0.031949 -0.242846
  0.745432 -0.620771  0.983217  0.163156 -0.081632 -0.003638 -0.429826
 -0.902904  0.998294 -0.055238 -0.018922  0.052290  0.989972 -0.131229
  0.999972  0.007316  0.001436 -0.007318  0.999972  0.001334  0.999981
  0.005496  0.002717 -0.005492  0.999984 -0.001497  0.991520 -0.128820
 -0.017140  0.126362  0.986485 -0.104307  0.999963 -0.008017  0.003030
  0.008036  0.999947 -0.006402  1.000000 -0.000360 -0.000566  0.000358
  0.999996 -0.002791  0.988737 -0.072831 -0.130745  0.051678  

---